# Week 3 — Customer Churn Prediction

Predicting customer churn using billing and tenure data, comparing a 
Decision Tree against Logistic Regression, and translating the findings 
into a business-ready recommendation.

### 1. Setup & Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

### 2. Load Dataset

In [2]:
df = pd.read_csv("../data/telco_churn.csv")
print(df.shape)
df.head()

(7043, 50)


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


### 3. Feature Selection

In [3]:
features = ['Tenure in Months', 'Monthly Charge', 'Contract', 'Payment Method', 'Internet Type', 'Total Charges']
target = 'Churn Label'

df_selected = df[features + [target]]
df_selected.head()

,Tenure in Months,Monthly Charge,Contract,Payment Method,Internet Type,Total Charges,Churn Label
0,1,39.65,Month-to-Month,Bank Withdrawal,DSL,39.65,Yes
1,8,80.65,Month-to-Month,Credit Card,Fiber Optic,633.30,Yes
2,18,95.45,Month-to-Month,Bank Withdrawal,Fiber Optic,1752.55,Yes
3,25,98.50,Month-to-Month,Bank Withdrawal,Fiber Optic,2514.50,Yes
4,37,76.50,Month-to-Month,Bank Withdrawal,Fiber Optic,2868.15,Yes


### 4. Exploratory Data Analysis (Churn Patterns)

In [4]:
print("Missing values:")
print(df_selected.isnull().sum())

print("\nChurn distribution:")
print(df_selected['Churn Label'].value_counts())

print("\nAvg tenure by churn:")
print(df_selected.groupby('Churn Label')['Tenure in Months'].mean())

print("\nAvg monthly charge by churn:")
print(df_selected.groupby('Churn Label')['Monthly Charge'].mean())

Missing values:
Tenure in Months       0
Monthly Charge         0
Contract               0
Payment Method         0
Internet Type       1526
Total Charges          0
Churn Label            0
dtype: int64

Churn distribution:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

Avg tenure by churn:
Churn Label
No     37.591225
Yes    17.979133
Name: Tenure in Months, dtype: float64

Avg monthly charge by churn:
Churn Label
No     61.265124
Yes    74.441332
Name: Monthly Charge, dtype: float64


### 5. Handle Missing Values

In [5]:
df_selected = df_selected.copy()
df_selected['Internet Type'] = df_selected['Internet Type'].fillna('No Internet Service')

print("Missing values after fix:")
print(df_selected.isnull().sum())

Missing values after fix:
Tenure in Months    0
Monthly Charge      0
Contract            0
Payment Method      0
Internet Type       0
Total Charges       0
Churn Label         0
dtype: int64


### 6. Encode Categorical Variables & Train-Test Split

In [6]:
# Encode categorical columns
df_encoded = pd.get_dummies(df_selected, columns=['Contract', 'Payment Method', 'Internet Type'], drop_first=True)

# Encode target
df_encoded['Churn'] = df_encoded['Churn Label'].map({'Yes': 1, 'No': 0})
df_encoded = df_encoded.drop('Churn Label', axis=1)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (5634, 10)
Test set: (1409, 10)


### 7. Train & Compare Models (Logistic Regression vs. Decision Tree)

In [7]:
# Logistic Regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)

# Decision Tree
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Decision Tree Accuracy:", accuracy_score(y_test, tree_pred))

print("\nLogistic Regression Report:")
print(classification_report(y_test, log_pred))

print("\nDecision Tree Report:")
print(classification_report(y_test, tree_pred))

Logistic Regression Accuracy: 0.8026969481902059
Decision Tree Accuracy: 0.7395315826827538

Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.85      0.88      0.86      1009
           1       0.66      0.61      0.64       400

    accuracy                           0.80      1409
   macro avg       0.76      0.75      0.75      1409
weighted avg       0.80      0.80      0.80      1409


Decision Tree Report:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1009
           1       0.54      0.53      0.54       400

    accuracy                           0.74      1409
   macro avg       0.68      0.68      0.68      1409
weighted avg       0.74      0.74      0.74      1409



/home/pronomanrizvi/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 8. Feature Importance (Top Churn Drivers)

In [8]:
importances = pd.Series(tree_model.feature_importances_, index=X.columns)
top_3 = importances.sort_values(ascending=False).head(3)

print("Top 3 Features Driving Churn:")
print(top_3)

Top 3 Features Driving Churn:
Monthly Charge      0.286820
Total Charges       0.285259
Tenure in Months    0.214812
dtype: float64


### Model Comparison & Key Findings

**Class Imbalance Note:** The dataset has a moderate class imbalance — 5,174 
customers did not churn vs. 1,869 who did (~73.5% vs. 26.5%). This wasn't 
explicitly corrected (e.g., no oversampling/undersampling), but is worth 
noting since it affects recall on the minority (churned) class in both models.

**Model Performance:**

| Metric | Logistic Regression | Decision Tree |
|---|---|---|
| Accuracy | 80.3% | 74.0% |
| Precision (Churn) | 0.66 | 0.54 |
| Recall (Churn) | 0.61 | 0.53 |
| F1-score (Churn) | 0.64 | 0.54 |

Logistic Regression outperformed the Decision Tree across every metric. This 
is common with an untuned Decision Tree, which tends to overfit the training 
data and generalize less well to new customers.

**Top 3 Features Driving Churn (from Decision Tree):**
1. **Monthly Charge** — higher bills correlate strongly with churn
2. **Total Charges** — related to overall spend/lifetime value
3. **Tenure in Months** — newer customers churn more than long-tenured ones

### Business Summary

We looked at what's driving customer churn and found a clear pattern: 
customers who leave tend to be newer (averaging 18 months with us, versus 
38 months for customers who stay) and pay noticeably higher monthly bills 
($74 versus $61 on average). About 1 in 4 customers in our data churned, 
so this is a meaningful chunk of revenue at risk. Our model, using billing 
and tenure information, can flag likely churners with reasonable accuracy 
(80%), which means we could proactively reach out to at-risk customers — 
especially newer, higher-paying ones — before they leave. The next step 
would be testing retention offers (like discounts or contract incentives) 
specifically targeted at this group.